In [1]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "johnson16-2-4.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: johnson16-2-4.clq
8
5 13 33 43 56 86 104 116
0.03sec 9.46sec
-----
Total time for johnson16-2-4.clq: 9.49sec



In [2]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "johnson8-2-4.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: johnson8-2-4.clq
4
5 13 16 26
0.00sec 0.00sec
-----
Total time for johnson8-2-4.clq: 0.00sec



In [3]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "keller4.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: keller4.clq
11
2 6 16 22 61 74 84 110 117 124 140
0.04sec 8.69sec
-----
Total time for keller4.clq: 8.73sec



In [5]:
import time
import random

def read_dimacs(filename):
    graph = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line[0] == 'c':
                continue
            parts = line.split()
            if parts[0] == 'p':
                n = int(parts[2])
                graph = [set() for _ in range(n)]
            elif parts[0] == 'e':
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                graph[u].add(v)
                graph[v].add(u)
    return graph

def greedy_clique_from(graph, start):
    clique = {start}
    candidates = list(graph[start])
    random.shuffle(candidates)
    for v in candidates:
        if all(v in graph[u] for u in clique):
            clique.add(v)
            candidates = [w for w in candidates if w in graph[v]]
    return clique

def heuristic_max_clique(graph, iterations=100):
    n = len(graph)
    best_clique = set()
    vertices = list(range(n))
    random.shuffle(vertices)
    for v in vertices:
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    for _ in range(iterations):
        v = random.choice(vertices)
        clique = greedy_clique_from(graph, v)
        if len(clique) > len(best_clique):
            best_clique = clique
    return best_clique

def greedy_coloring(graph, candidates):
    n = len(candidates)
    colors = [0] * n
    max_color = 0
    for i, v in enumerate(candidates):
        forbidden = set()
        for j in range(i):
            u = candidates[j]
            if u in graph[v]:
                forbidden.add(colors[j])
        col = 1
        while col in forbidden:
            col += 1
        colors[i] = col
        max_color = max(max_color, col)
    return max_color

def branch_and_bound(graph, candidates, current_clique, best_clique, start_time, time_limit):
    if time.time() - start_time >= time_limit:
        return best_clique
    if not candidates:
        if len(current_clique) > len(best_clique):
            return current_clique.copy()
        return best_clique

    color_bound = greedy_coloring(graph, candidates)
    if len(current_clique) + color_bound <= len(best_clique):
        return best_clique

    for i, v in enumerate(candidates):
        if time.time() - start_time >= time_limit:
            break
        new_clique = current_clique.copy()
        new_clique.add(v)
        new_candidates = [w for w in candidates[i+1:] if w in graph[v]]
        best_clique = branch_and_bound(graph, new_candidates, new_clique, best_clique, start_time, time_limit)
    return best_clique

def main():
    test_files = [
        "MANN_a27.clq"
    ]

    total_time_limit = 7200.0

    for filename in test_files:
        print("Testing file:", filename)
        graph = read_dimacs(filename)
        overall_start = time.time()

        heuristic_start = time.time()
        initial_clique = heuristic_max_clique(graph, iterations=100)
        heuristic_time = time.time() - heuristic_start

        bnb_time_limit = max(0, total_time_limit - heuristic_time)
        bnb_start = time.time()
        candidates = list(range(len(graph)))
        best_clique = initial_clique.copy()
        best_clique = branch_and_bound(graph, candidates, set(), best_clique, bnb_start, bnb_time_limit)
        bnb_time = time.time() - bnb_start

        final_clique = best_clique
        clique_vertices = sorted(list(final_clique))

        print(len(final_clique))
        print(" ".join(str(v + 1) for v in clique_vertices))
        print(f"{heuristic_time:.2f}sec {bnb_time:.2f}sec")
        print("-----")

        overall_time = time.time() - overall_start
        print(f"Total time for {filename}: {overall_time:.2f}sec\n")

if __name__ == "__main__":
    main()

Testing file: MANN_a27.clq
125
1 2 3 4 5 6 10 11 12 13 14 15 34 39 42 45 48 51 54 57 60 63 70 75 78 81 84 87 90 93 96 99 100 103 106 109 112 115 118 121 124 127 130 133 138 141 142 147 150 153 156 159 162 165 168 171 174 177 178 182 185 188 191 194 197 200 203 206 210 213 214 219 222 225 228 231 234 237 240 243 246 249 250 254 257 260 263 266 269 272 275 278 282 285 286 289 292 295 298 301 304 307 310 313 318 321 322 325 328 331 334 337 340 343 346 349 354 357 360 363 366 369 370 373 376
5.48sec 7194.52sec
-----
Total time for MANN_a27.clq: 7200.00sec

